# Transpilation

This notebook covers transpiling circuits for hardware targets, comparing
optimization levels, and circuit composition utilities.

In [ ]:
import (
	"context"
	"fmt"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/transpile/pipeline"
	"github.com/splch/goqu/transpile/target"
)

In [ ]:
var (
	c          *ir.Circuit
	transpiled *ir.Circuit
	ctx        = context.Background()
)

## Transpiling for Hardware

Build a circuit with a Toffoli gate (CCX) that needs decomposition for
hardware targets that only support 1- and 2-qubit basis gates.

In [ ]:
%%
c, _ = builder.New("toffoli", 3).H(2).CCX(0, 1, 2).H(2).Build()
gonbui.DisplaySvg(draw.SVG(c))

In [ ]:
%%
transpiled, _ = pipeline.Run(ctx, c, target.IBMBrisbane, pipeline.LevelFull)
fmt.Printf("Before: depth=%d gates=%d 2q=%d\n",
	c.Stats().Depth, c.Stats().GateCount, c.Stats().TwoQubitGates)
fmt.Printf("After:  depth=%d gates=%d 2q=%d\n",
	transpiled.Stats().Depth, transpiled.Stats().GateCount, transpiled.Stats().TwoQubitGates)

## Comparing Optimization Levels

Goqu provides four optimization levels:
- **LevelNone (0):** Decompose only
- **LevelBasic (1):** + cancel + merge
- **LevelFull (2):** + commute + parallelize
- **LevelParallel (3):** Multi-strategy, pick best

In [ ]:
%%
levels := []struct {
	name  string
	level pipeline.Level
}{
	{"None", pipeline.LevelNone},
	{"Basic", pipeline.LevelBasic},
	{"Full", pipeline.LevelFull},
	{"Parallel", pipeline.LevelParallel},
}

for _, l := range levels {
	out, _ := pipeline.Run(ctx, c, target.IBMBrisbane, l.level)
	s := out.Stats()
	fmt.Printf("%-10s depth=%-3d gates=%-3d 2q=%-3d\n", l.name, s.Depth, s.GateCount, s.TwoQubitGates)
}

## Before / After SVG

Display the original and transpiled circuits side by side using inline HTML.

In [ ]:
%%
html := fmt.Sprintf(
	`<div style="display:flex;gap:2em"><div><h4>Original</h4>%s</div><div><h4>Transpiled (LevelFull)</h4>%s</div></div>`,
	draw.SVG(c), draw.SVG(transpiled),
)
gonbui.DisplayHTML(html)

## Circuit Composition

Goqu provides utilities to compose circuits:
- `ir.Compose()` — append one circuit after another
- `ir.Tensor()` — place circuits on disjoint qubit spaces
- `ir.Inverse()` — reverse and adjoint a circuit

In [ ]:
%%
a, _ := builder.New("a", 2).H(0).CNOT(0, 1).Build()
b, _ := builder.New("b", 2).X(0).Z(1).Build()

composed, _ := ir.Compose(a, b, nil, nil)
fmt.Printf("Compose: %d qubits, depth %d\n", composed.NumQubits(), composed.Stats().Depth)
gonbui.DisplaySvg(draw.SVG(composed))

In [ ]:
%%
a2, _ := builder.New("a", 2).H(0).CNOT(0, 1).Build()
b2, _ := builder.New("b", 2).X(0).Z(1).Build()

tensored := ir.Tensor(a2, b2)
fmt.Printf("Tensor: %d qubits, depth %d\n", tensored.NumQubits(), tensored.Stats().Depth)
gonbui.DisplaySvg(draw.SVG(tensored))

In [ ]:
%%
a3, _ := builder.New("a", 2).H(0).CNOT(0, 1).Build()
inv := ir.Inverse(a3)
fmt.Printf("Inverse: %d qubits, depth %d\n", inv.NumQubits(), inv.Stats().Depth)
gonbui.DisplaySvg(draw.SVG(inv))